# Learn++ Implementation from Scratch

This notebook implements the Learn++ algorithm (Polikar et al., 2001) step by step.
Each section maps directly to the equations in `planning_seed/06_LEARNPP_ALGORITHM.md`.

## Why from scratch?
- Full understanding of internals for presentation and report
- Step-by-step comments explaining each decision
- Easy to modify for our specific comparison (Decision Tree vs Gradient Boosting)

## Hardware
- Local: RTX A1000 + i9 (sufficient for this scale)

## Dependencies
```
numpy, scikit-learn, matplotlib, seaborn
```

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Learn++ Algorithm Overview

Learn++ is an **ensemble-based incremental learning** algorithm. It:
1. Receives data in sequential batches (D₁, D₂, D₃, ...)
2. Trains multiple weak hypotheses on each batch
3. Combines all hypotheses via weighted majority voting
4. Never discards old hypotheses — knowledge accumulates

### Key Idea
Each batch trains T_k base learners. Samples that are hard to classify get higher weights
(like AdaBoost), but across batches the ensemble only grows — it never forgets.

### Core Equations
| Symbol | Meaning |
|--------|---------|
| D_k | k-th data batch |
| T_k | Number of base learners trained on batch k |
| h_t | Base hypothesis (classifier) at iteration t |
| ε_t | Weighted error of h_t |
| β_t | Confidence weight = ε_t / (1 - ε_t) |
| w_i | Sample weight for instance i |
| H_final | Final ensemble (weighted majority vote) |

## 2. Base Learner Definitions

We compare two base learners within Learn++:

| Learner | Type | Key Params | Why |
|---------|------|-----------|-----|
| Decision Tree (depth=2) | Simple | max_depth=2 | Canonical Learn++ base learner from original paper |
| Gradient Boosting | Complex | n_estimators=50, max_depth=3 | Same tree family, much higher capacity |

In [ ]:
def make_simple_learner():
    """Decision Tree, depth 2 — the canonical Learn++ base learner.
    
    Why depth 2?
    - Original Polikar et al. (2001) used shallow decision stumps/trees
    - Weak enough that the ensemble mechanism provides the power
    - Fast to train, low memory footprint
    """
    return DecisionTreeClassifier(
        max_depth=2,
        random_state=RANDOM_SEED
    )


def make_complex_learner():
    """Gradient Boosting — complex base learner for capacity comparison.
    
    Why Gradient Boosting?
    - Same tree-based family as Decision Tree (isolates capacity variable)
    - Much higher capacity: ensemble of 50 trees, each depth 3
    - Question: does Learn++ benefit from a strong base learner,
      or does the extra capacity cause overfitting within batches?
    """
    return GradientBoostingClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        random_state=RANDOM_SEED
    )

## 3. Learn++ Implementation

### Step-by-step:
1. **Initialize** sample weights uniformly
2. **For each iteration t** in batch k:
   - Draw a weighted subset from the batch (using sample weights as distribution)
   - Train base learner h_t on the subset
   - Compute weighted error ε_t on the FULL batch
   - If ε_t ≥ 0.5: discard h_t, re-initialize weights, retry
   - Compute confidence: β_t = ε_t / (1 - ε_t)
   - Update sample weights: reduce weights of correctly classified samples
   - Normalize weights
3. **After all batches**: final prediction = weighted majority vote across all h_t

In [ ]:
class LearnPP:
    """Learn++ incremental learning algorithm.
    
    Reference: Polikar, R., Upda, L., Upda, S.S., & Honavar, V. (2001).
    Learn++: An incremental learning algorithm for supervised neural networks.
    IEEE Transactions on Systems, Man, and Cybernetics.
    """
    
    def __init__(self, base_learner_factory, T_k=10, max_retries=3):
        """
        Args:
            base_learner_factory: Callable that returns a fresh base learner instance.
            T_k: Number of base learners to train per batch.
            max_retries: Max attempts if a hypothesis has error >= 0.5.
        """
        self.base_learner_factory = base_learner_factory
        self.T_k = T_k
        self.max_retries = max_retries
        
        # Accumulated ensemble across all batches
        self.hypotheses = []   # List of trained base learners
        self.betas = []        # Confidence weight for each hypothesis
        self.classes_ = None   # Known class labels
    
    def partial_fit(self, X, y):
        """Train on a new batch of data (incremental learning step).
        
        This is called once per batch D_k. The ensemble grows by T_k hypotheses.
        Old hypotheses are NEVER removed — this is how Learn++ avoids forgetting.
        
        Args:
            X: Feature matrix for this batch, shape (n_samples, n_features)
            y: Labels for this batch, shape (n_samples,)
        """
        n_samples = len(X)
        
        # Track all known classes across batches
        if self.classes_ is None:
            self.classes_ = np.unique(y)
        else:
            self.classes_ = np.unique(np.concatenate([self.classes_, np.unique(y)]))
        
        # --- Eq. 1: Initialize sample weights uniformly ---
        # Every sample starts equally important
        w = np.ones(n_samples) / n_samples
        
        for t in range(self.T_k):
            for retry in range(self.max_retries):
                # --- Eq. 2: Create training distribution D_t from weights ---
                # Draw a weighted bootstrap sample
                # Samples with higher weights are more likely to be selected
                indices = np.random.choice(
                    n_samples, size=n_samples, replace=True, p=w
                )
                X_train, y_train = X[indices], y[indices]
                
                # --- Eq. 3: Train base hypothesis h_t ---
                h_t = self.base_learner_factory()
                h_t.fit(X_train, y_train)
                
                # --- Eq. 4: Compute weighted error on FULL batch ---
                predictions = h_t.predict(X)
                incorrect = (predictions != y).astype(float)
                epsilon_t = np.dot(w, incorrect)  # Weighted error
                
                # --- Eq. 5: Check error condition ---
                # If error >= 0.5, this hypothesis is no better than random
                if epsilon_t < 0.5:
                    break  # Good hypothesis, proceed
                else:
                    # Reset weights and retry
                    w = np.ones(n_samples) / n_samples
            else:
                # All retries failed — skip this iteration
                continue
            
            # --- Eq. 6: Compute confidence weight ---
            # beta_t = epsilon_t / (1 - epsilon_t)
            # Lower error → lower beta → higher confidence (log(1/beta) is the vote weight)
            beta_t = epsilon_t / (1 - epsilon_t + 1e-10)  # Small epsilon for stability
            
            # Store hypothesis and its confidence
            self.hypotheses.append(h_t)
            self.betas.append(beta_t)
            
            # --- Eq. 7: Update sample weights ---
            # Correctly classified samples get REDUCED weight
            # (so next iteration focuses on harder samples)
            correct = (predictions == y).astype(float)
            w = w * (beta_t ** correct)
            
            # --- Eq. 8: Normalize weights ---
            w = w / w.sum()
    
    def predict(self, X):
        """Predict using weighted majority vote across ALL hypotheses.
        
        Each hypothesis votes for a class. Votes are weighted by log(1/beta).
        The class with the highest total vote wins.
        """
        if not self.hypotheses:
            raise RuntimeError("No hypotheses trained yet. Call partial_fit first.")
        
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        
        # Vote accumulator: shape (n_samples, n_classes)
        votes = np.zeros((n_samples, n_classes))
        
        for h_t, beta_t in zip(self.hypotheses, self.betas):
            # Vote weight = log(1 / beta_t)
            # Lower beta (better classifier) = higher vote weight
            vote_weight = np.log(1.0 / (beta_t + 1e-10))
            
            predictions = h_t.predict(X)
            for i, pred in enumerate(predictions):
                class_idx = np.where(self.classes_ == pred)[0]
                if len(class_idx) > 0:
                    votes[i, class_idx[0]] += vote_weight
        
        # Final prediction = class with highest accumulated vote
        return self.classes_[np.argmax(votes, axis=1)]

## 4. Quick Validation on Synthetic Data

Before running on Fashion-MNIST or BraTS, let's verify the implementation works
on a simple 2D classification problem with incremental batches.

In [ ]:
from sklearn.datasets import make_classification

# Create a simple dataset split into 3 batches
X_full, y_full = make_classification(
    n_samples=900, n_features=10, n_classes=3,
    n_informative=6, random_state=RANDOM_SEED
)

# Split into 3 incremental batches
batches = [
    (X_full[:300], y_full[:300]),    # D1
    (X_full[300:600], y_full[300:600]),  # D2
    (X_full[600:], y_full[600:]),    # D3
]

# Holdout test set
X_test, y_test = make_classification(
    n_samples=300, n_features=10, n_classes=3,
    n_informative=6, random_state=RANDOM_SEED + 1
)

print(f"Batches: {[b[0].shape[0] for b in batches]} samples each")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Run Learn++ with Decision Tree (simple) base learner
model_simple = LearnPP(base_learner_factory=make_simple_learner, T_k=10)

print("=== Learn++ with Decision Tree (depth=2) ===")
for k, (X_batch, y_batch) in enumerate(batches, 1):
    model_simple.partial_fit(X_batch, y_batch)
    y_pred = model_simple.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    ba = balanced_accuracy_score(y_test, y_pred)
    print(f"  After D{k}: MacroF1={f1:.3f}, BalancedAcc={ba:.3f}, "
          f"Ensemble size={len(model_simple.hypotheses)}")

In [ ]:
# Run Learn++ with Gradient Boosting (complex) base learner
model_complex = LearnPP(base_learner_factory=make_complex_learner, T_k=10)

print("=== Learn++ with Gradient Boosting ===")
for k, (X_batch, y_batch) in enumerate(batches, 1):
    model_complex.partial_fit(X_batch, y_batch)
    y_pred = model_complex.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    ba = balanced_accuracy_score(y_test, y_pred)
    print(f"  After D{k}: MacroF1={f1:.3f}, BalancedAcc={ba:.3f}, "
          f"Ensemble size={len(model_complex.hypotheses)}")

## 5. Next Steps

Once this validation passes:
1. **EXP-01**: Load Fashion-MNIST, design batches per `04_DATA_STRATEGY.md`, run comparison
2. **EXP-02**: Load BraTS, extract ROI features, run comparison
3. **Metrics**: Compute full CompositeScore (quality + cost)
4. **Statistics**: Wilcoxon signed-rank test for significance

See `planning_seed/03_EXPERIMENT_PLAN.md` for the full protocol.